# 09. MAF でマルチエージェント化する

## コードの解説

MAF の `HandoffBuilder` を使ってマルチエージェントシステムを構築します。
**コーディネーターエージェント**がユーザーの意図を分析し、最適な**専門エージェント**にハンドオフします。
各専門エージェントはそのドメインに特化した処理を担当します。

### HandoffBuilder の仕組み

```
ユーザー質問
    ↓
コーディネーターエージェント（ルーティング判断）
    ├─→ 製品担当エージェント（製品に関する質問）
    ├─→ 注文担当エージェント（注文・配送に関する質問）
    └─→ サポート担当エージェント（技術的な問題）
```

### マルチエージェントを使う理由

| シングルエージェント | マルチエージェント |
|------------------|----------------|
| すべての責任を 1 つのエージェントが持つ | 役割を分担して専門性を高める |
| 複雑なタスクで精度が落ちやすい | 各エージェントが得意分野に集中 |
| スケールしにくい | 新しい専門エージェントを追加するだけ |

### HandoffBuilder の設定パラメータ

| パラメータ | 説明 |
|---------|------|
| `coordinator` | ルーティングを担当するコーディネーターエージェント |
| `participants` | ハンドオフ先の専門エージェントのリスト |

In [ ]:
import os
from dotenv import load_dotenv
from agent_framework import Agent, MCPStreamableHTTPTool
from agent_framework.openai import OpenAIChatCompletionClient
from agent_framework.orchestrations import HandoffBuilder

load_dotenv()

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY  = os.getenv("AZURE_OPENAI_API_KEY")
MODEL_DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
MCP_SERVER_URI        = os.getenv("MCP_SERVER_URI", "http://localhost:8000/mcp")

USE_KEY_AUTH = os.getenv("USE_KEY_AUTH", "False").lower() in ("true", "1", "t")
if USE_KEY_AUTH:
    auth_kwargs = dict(api_key=AZURE_OPENAI_API_KEY)
else:
    from azure.identity.aio import DefaultAzureCredential
    os.environ.pop("AZURE_OPENAI_API_KEY", None)
    auth_kwargs = dict(credential=DefaultAzureCredential())

# LLM クライアント（全エージェント共有）
chat_client = OpenAIChatCompletionClient(
    **auth_kwargs,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    model=MODEL_DEPLOYMENT_NAME,
)

# MCP ツール（製品・注文情報へのアクセス）
mcp_tool = MCPStreamableHTTPTool(
    name="game_shop_tools",
    url=MCP_SERVER_URI,
    description="Xbox ゲームショップの製品・注文・在庫情報を取得するツール群です。",
    timeout=120,
)


# ---------------------------------------------------------------
# 専門エージェントの定義
# 各エージェントは自分のドメインに特化した instructions を持つ
# ---------------------------------------------------------------
product_agent = Agent(
    client=chat_client,
    name="ProductSpecialist",
    description="製品情報・価格・在庫に関する質問を担当します。",
    instructions=(
        "あなたは Xbox ゲームショップの製品スペシャリストです。"
        "MCP ツールを使って製品情報、価格、在庫を調べ、的確な情報を提供してください。"
    ),
    tools=[mcp_tool],
)

order_agent = Agent(
    client=chat_client,
    name="OrderSpecialist",
    description="注文状況・配送・返品に関する質問を担当します。",
    instructions=(
        "あなたは Xbox ゲームショップの注文スペシャリストです。"
        "MCP ツールを使って注文情報を調べ、配送状況や返品手続きをサポートしてください。"
    ),
    tools=[mcp_tool],
)


# ---------------------------------------------------------------
# コーディネーターエージェントの定義
# ユーザーの質問を分析し、適切な専門エージェントにハンドオフする
# ---------------------------------------------------------------
coordinator = Agent(
    client=chat_client,
    name="Coordinator",
    description="ユーザーの質問を分析し、適切な専門エージェントに振り分けます。",
    instructions=(
        "あなたは Xbox ゲームショップのコーディネーターです。"
        "ユーザーの質問を分析し、製品に関する質問は ProductSpecialist に、"
        "注文・配送に関する質問は OrderSpecialist にハンドオフしてください。"
    ),
)


# ---------------------------------------------------------------
# HandoffBuilder でマルチエージェントワークフローを構築
# coordinator が判断し、participants にハンドオフする
# ---------------------------------------------------------------
workflow = (
    HandoffBuilder()
    .with_coordinator(coordinator)
    .with_participants([product_agent, order_agent])
    .build()
)

# マルチエージェントワークフローの実行
print("=== マルチエージェントワークフロー ===")
response = await workflow.run("Xbox Series X の在庫はありますか？")
print(response)